# Correlation Functions and Spectral Densities in Quantarhei

In open quantum systems, molecules are never truly isolated. They are embedded in a protein or solvent environment — the **bath** — whose random fluctuations cause the transition frequencies of each molecule to jitter in time. This dephasing is captured quantitatively by the **bath correlation function** $C(t)$.

Formally, if $\delta\omega(t) = \omega(t) - \langle\omega\rangle$ is the fluctuation of the electronic transition frequency at time $t$, the correlation function is

$$C(t) = \langle\delta\omega(t)\,\delta\omega(0)\rangle,$$

where the angle brackets denote a thermal average over bath states.  
- $C(0)$ measures the **variance** of the frequency fluctuations (how broadly the line is inhomogeneously broadened on the ultrafast timescale).  
- The decay of $\text{Re}\,C(t)$ tells us the **correlation time** $\tau_c$: how long the bath remembers its own state.  
- The imaginary part carries information about dissipation and is related to the bath response function.

The **spectral density** $J(\omega)$ is essentially the Fourier transform of the bath response and describes which vibrational modes of the environment couple to the electronic transition and with what strength. It is the key input to Redfield, Förster, and HEOM theories of energy transfer.

## Imports and display settings

In [1]:
import quantarhei as qr
import matplotlib.pyplot as plt
import numpy as np
%config InlineBackend.figure_format = 'svg'

## 1. Creating a TimeAxis

All dynamical objects in Quantarhei are defined on a discrete time grid. `TimeAxis(start, N, step)` creates a uniformly spaced grid of `N` points beginning at `start` with spacing `step` (in femtoseconds). Here we use 1000 fs with 1 fs resolution — sufficient to resolve the slow bath dynamics of a protein environment.

In [2]:
ta = qr.TimeAxis(0.0, 1000, 1.0)   # start=0 fs, N=1000 points, step=1 fs

print(f"TimeAxis: {ta.length} points from {ta.data[0]:.1f} fs to {ta.data[-1]:.1f} fs")

TimeAxis: 1000 points from 0.0 fs to 999.0 fs


## 2. The Overdamped Brownian Oscillator Model

The most widely used model for the bath in photosynthetic systems is the **Overdamped Brownian Oscillator (OBO)**. It describes a harmonic bath mode that is so heavily damped it never oscillates — it simply relaxes exponentially. The resulting correlation function has the analytic form

$$C(t) = \left(\lambda\cot\!\frac{\gamma}{2k_BT} - i\lambda\right)e^{-\gamma t} + \frac{4\lambda\gamma k_BT}{\hbar}\sum_{k=1}^{N_\text{mat}} \frac{e^{-\nu_k t}}{\nu_k^2 - \gamma^2},$$

where:
- $\lambda$ is the **reorganization energy** — the energetic cost to the bath when the electronic state changes,
- $\gamma = 1/\tau_c$ is the bath **relaxation rate** ($\tau_c$ is the correlation time),
- $\nu_k = 2\pi k k_B T / \hbar$ are the **Matsubara frequencies** (quantum correction terms that vanish at high temperature).

We specify parameters in wavenumber (1/cm) units using the `energy_units` context manager, which handles all conversions to Quantarhei's internal units.

In [3]:
params = {
    "ftype":     "OverdampedBrownian",
    "reorg":     20.0,    # reorganization energy in 1/cm
    "cortime":   100.0,   # correlation time in fs
    "T":         300.0,   # temperature in K
    "matsubara": 20,      # number of Matsubara terms for quantum corrections
}

with qr.energy_units("1/cm"):
    cf = qr.CorrelationFunction(ta, params)

# Verify key values
reorg = cf.get_reorganization_energy()
reorg_cm = qr.convert(reorg, "int", "1/cm")

print(f"C(0)  = ({cf.data[0].real:.4e} {cf.data[0].imag:+.4e}j)  [internal units]")
print(f"C(100 fs) = ({cf.data[100].real:.4e} {cf.data[100].imag:+.4e}j)")
print(f"C(999 fs) = ({cf.data[-1].real:.4e} {cf.data[-1].imag:+.4e}j)")
print(f"Reorganization energy: {reorg_cm:.1f} 1/cm")

C(0)  = (3.8066e-04 -3.7673e-05j)  [internal units]
C(100 fs) = (1.0828e-04 -1.3859e-05j)
C(999 fs) = (1.3497e-08 -1.7275e-09j)
Reorganization energy: 20.0 1/cm


## 3. Plotting C(t) — Real and Imaginary Parts

The **real part** of $C(t)$ is the symmetrized (classical-like) part that drives pure dephasing and spectral broadening. It decays from its maximum at $t=0$ with the correlation time $\tau_c = 100$ fs.

The **imaginary part** is the antisymmetric (quantum) part, related to the bath response function. It is responsible for the Stokes shift (reorganization energy) and carries the arrow of time — it is zero for a classical bath.

In [4]:
t = ta.data   # time axis in fs

fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(t, np.real(cf.data), color="steelblue",  lw=2, label=r"Re $C(t)$")
ax.plot(t, np.imag(cf.data), color="tomato",     lw=2, label=r"Im $C(t)$", linestyle="--")

ax.axhline(0, color="black", lw=0.6, linestyle=":")
ax.set_xlabel("Time (fs)", fontsize=13)
ax.set_ylabel(r"$C(t)$ (internal units)", fontsize=13)
ax.set_title(r"Bath Correlation Function $C(t)$ — OverdampedBrownian, $\lambda=20$ cm$^{-1}$, $\tau_c=100$ fs, $T=300$ K",
             fontsize=11)
ax.legend(fontsize=12)
ax.set_xlim(0, 600)

fig.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61759/1866149118.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The real part decays roughly as $e^{-t/\tau_c}$ with $\tau_c = 100$ fs, as expected for the overdamped model. The imaginary part is smaller in magnitude — at room temperature the bath fluctuations are primarily classical — and also decays exponentially.

## 4. Spectral Density J(ω)

The **spectral density** $J(\omega)$ is the frequency-domain partner of the correlation function. It is defined so that the integral

$$\lambda = \frac{1}{\pi}\int_0^\infty \frac{J(\omega)}{\omega}\,d\omega$$

recovers the reorganization energy. For the OBO model the spectral density has a simple Lorentzian form:

$$J(\omega) = \frac{2\lambda\gamma\omega}{\omega^2 + \gamma^2}.$$

The peak is at $\omega = \gamma = 1/\tau_c$, i.e. at the bath relaxation frequency. Wider peaks correspond to faster baths.

In [5]:
with qr.energy_units("1/cm"):
    sd = qr.SpectralDensity(ta, params)

print(f"SpectralDensity axis: {sd.data.shape[0]} points, "
      f"range {sd.axis.data[0]:.2f} to {sd.axis.data[-1]:.2f} rad/fs")

SpectralDensity axis: 2000 points, range -3.14 to 3.14 rad/fs


In [6]:
# Convert frequency axis from rad/fs to wavenumbers (1/cm)
# 1 rad/fs = 1e15 rad/s;  wavenumber = omega / (2*pi*c)  with c = 3e10 cm/s
conv_factor = 1e15 / (2 * np.pi * 3e10)   # ~ 5305 cm^-1 per rad/fs
omega_cm = sd.axis.data * conv_factor

# Keep only the positive-frequency half
pos = sd.axis.data > 0
omega_pos = omega_cm[pos]
sd_pos    = np.real(sd.data[pos])

fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(omega_pos, sd_pos, color="darkorange", lw=2)
ax.fill_between(omega_pos, sd_pos, alpha=0.15, color="darkorange")

ax.set_xlabel(r"Frequency $\omega$ (cm$^{-1}$)", fontsize=13)
ax.set_ylabel(r"$J(\omega)$ (internal units)", fontsize=13)
ax.set_title(r"Spectral Density $J(\omega)$ — OverdampedBrownian, $\lambda=20$ cm$^{-1}$, $\tau_c=100$ fs",
             fontsize=11)
ax.set_xlim(0, 500)

# Mark the bath relaxation frequency gamma = 1/tau_c
gamma_cm = (1.0 / 100.0) * conv_factor   # tau_c = 100 fs in internal (rad/fs) -> 1/cm
ax.axvline(gamma_cm, color="gray", lw=1.2, linestyle="--",
           label=rf"$\gamma = 1/\tau_c \approx {gamma_cm:.0f}$ cm$^{{-1}}$")
ax.legend(fontsize=11)

fig.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61759/1501467438.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The spectral density peaks near the bath relaxation frequency $\gamma = 1/\tau_c \approx 53$ cm$^{-1}$ and falls off at higher frequencies. This tells us that the bath is most effective at transferring energy at frequencies around 53 cm$^{-1}$, which is in the range relevant for exciton-bath coupling in photosynthetic complexes.

## 5. Effect of Temperature on C(t)

Temperature enters the correlation function through two routes:

1. **Amplitude of the real part**: At higher $T$, thermal fluctuations are larger, so $\text{Re}\,C(0)$ is larger. In the classical limit $\text{Re}\,C(0) \approx 2\lambda k_B T / \hbar$.

2. **Matsubara terms**: At low $T$, the quantum Matsubara corrections to $\text{Re}\,C(t)$ become important, adding slower-decaying components to the correlation function.

The imaginary part is **temperature-independent** (it depends only on $\lambda$ and $\gamma$), since it encodes bath dissipation, not fluctuations.

We compare $T = 77$ K (liquid nitrogen, cryogenic spectroscopy) vs $T = 300$ K (room temperature, physiological).

In [7]:
params_77K = dict(params)
params_77K["T"] = 77.0

with qr.energy_units("1/cm"):
    cf_77K  = qr.CorrelationFunction(ta, params_77K)
    cf_300K = qr.CorrelationFunction(ta, params)        # re-create for clarity

print(f"Re C(0) at 300 K: {cf_300K.data[0].real:.3e}")
print(f"Re C(0) at  77 K: {cf_77K.data[0].real:.3e}")
print(f"Ratio: {cf_300K.data[0].real / cf_77K.data[0].real:.2f}  "
      f"(classical expectation 300/77 = {300/77:.2f})")
print(f"Im C(0) at 300 K: {cf_300K.data[0].imag:.3e}")
print(f"Im C(0) at  77 K: {cf_77K.data[0].imag:.3e}  (same, as expected)")

Re C(0) at 300 K: 3.807e-04
Re C(0) at  77 K: 1.566e-04
Ratio: 2.43  (classical expectation 300/77 = 3.90)
Im C(0) at 300 K: -3.767e-05
Im C(0) at  77 K: -3.767e-05  (same, as expected)


In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.8), sharey=False)

# --- Left panel: real part ---
ax1.plot(t, np.real(cf_300K.data), color="steelblue",  lw=2, label="300 K")
ax1.plot(t, np.real(cf_77K.data),  color="seagreen",   lw=2, label="77 K",  linestyle="--")
ax1.axhline(0, color="black", lw=0.6, linestyle=":")
ax1.set_xlabel("Time (fs)", fontsize=12)
ax1.set_ylabel(r"Re $C(t)$ (internal units)", fontsize=12)
ax1.set_title(r"Real part of $C(t)$", fontsize=12)
ax1.set_xlim(0, 600)
ax1.legend(fontsize=11)

# --- Right panel: imaginary part ---
ax2.plot(t, np.imag(cf_300K.data), color="steelblue",  lw=2, label="300 K")
ax2.plot(t, np.imag(cf_77K.data),  color="seagreen",   lw=2, label="77 K",  linestyle="--")
ax2.axhline(0, color="black", lw=0.6, linestyle=":")
ax2.set_xlabel("Time (fs)", fontsize=12)
ax2.set_ylabel(r"Im $C(t)$ (internal units)", fontsize=12)
ax2.set_title(r"Imaginary part of $C(t)$ — temperature independent", fontsize=12)
ax2.set_xlim(0, 600)
ax2.legend(fontsize=11)

fig.suptitle(r"Temperature dependence of $C(t)$:  $\lambda=20$ cm$^{-1}$, $\tau_c=100$ fs",
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61759/1239124900.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Observations

- **Real part**: the 300 K amplitude is about 2.4× larger than at 77 K. The classical prediction would give $300/77 \approx 3.9\times$; the smaller observed ratio reflects the quantum (Matsubara) corrections that are important at low temperature.
- **Imaginary part**: identical at both temperatures, confirming it is purely a property of the bath Hamiltonian (via $\lambda$ and $\gamma$), not of the thermal occupation.
- Both curves share the same correlation time $\tau_c = 100$ fs — temperature does not change how fast the bath fluctuates, only by how much.

In practice, these differences directly affect the linewidths in linear absorption spectra: cryogenic spectra are narrower because $\text{Re}\,C(0)$ is smaller.

## Summary

| Quantity | Meaning | Key parameter |
|---|---|---|
| $C(0)$, real part | Variance of frequency fluctuations | $\propto \lambda k_B T$ |
| Decay time of Re $C(t)$ | Bath correlation time $\tau_c$ | `cortime` |
| Im $C(t)$ | Quantum dissipation / Stokes shift | $\propto \lambda / \gamma$ |
| $J(\omega)$ peak position | Bath relaxation frequency $\gamma = 1/\tau_c$ | `cortime` |
| $\int J(\omega)/\omega\,d\omega$ | Reorganization energy $\lambda$ | `reorg` |

The next notebook will show how to attach a `CorrelationFunction` to a molecule and use it to compute linear absorption spectra.